# Data Profiling — Workshop 1: Recruitment Data Warehouse

Initial exploration of the `candidates.csv` dataset to understand structure, quality, and key statistics.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/candidates.csv', sep=';', encoding='utf-8')
print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

Dataset loaded: 50,000 rows x 10 columns


## 1. Basic Structure

In [2]:
print('=== Column Names ===')
for i, col in enumerate(df.columns, 1):
    print(f'  {i}. {col}')

print(f'\n=== Shape: {df.shape} ===')
print(f'\n=== Data Types ===')
print(df.dtypes)

=== Column Names ===
  1. First Name
  2. Last Name
  3. Email
  4. Application Date
  5. Country
  6. YOE
  7. Seniority
  8. Technology
  9. Code Challenge Score
  10. Technical Interview Score

=== Shape: (50000, 10) ===

=== Data Types ===
First Name                     str
Last Name                      str
Email                          str
Application Date               str
Country                        str
YOE                          int64
Seniority                      str
Technology                     str
Code Challenge Score         int64
Technical Interview Score    int64
dtype: object


In [3]:
df.head(10)

,First Name,Last Name,Email,Application Date,Country,YOE,Seniority,Technology,Code Challenge Score,Technical Interview Score
0,Bernadette,Langworth,leonard91@yahoo.com,2021-02-26,Norway,2,Intern,Data Engineer,3,3
1,Camryn,Reynolds,zelda56@hotmail.com,2021-09-09,Panama,10,Intern,Data Engineer,2,10
2,Larue,Spinka,okey_schultz41@gmail.com,2020-04-14,Belarus,4,Mid-Level,Client Success,10,9
3,Arch,Spinka,elvera_kulas@yahoo.com,2020-10-01,Eritrea,25,Trainee,QA Manual,7,1
4,Larue,Altenwerth,minnie.gislason@gmail.com,2020-05-20,Myanmar,13,Mid-Level,Social Media Community Management,9,7
5,Alec,Abbott,juanita_hansen@gmail.com,2019-08-17,Zimbabwe,8,Junior,Adobe Experience Manager,2,9
6,Allison,Jacobs,alba_rolfson27@yahoo.com,2018-05-18,Wallis and Futuna,19,Trainee,Sales,2,9
7,Nya,Skiles,madisen.zulauf@gmail.com,2021-12-09,Myanmar,1,Lead,Mulesoft,2,5
8,Mose,Lakin,dale_murazik@hotmail.com,2018-03-13,Italy,18,Lead,Social Media Community Management,7,10
9,Terrance,Zieme,dustin31@hotmail.com,2022-04-08,Timor-Leste,25,Lead,DevOps,2,0


## 2. Missing Values

In [4]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])
if missing.sum() == 0:
    print('No missing values found.')

Empty DataFrame
Columns: [Missing Count, Missing %]
Index: []
No missing values found.


## 3. Duplicates

In [5]:
n_dup = df.duplicated().sum()
print(f'Duplicate rows: {n_dup:,} ({n_dup/len(df)*100:.2f}%)')
print(f'Unique rows: {len(df) - n_dup:,}')

Duplicate rows: 0 (0.00%)
Unique rows: 50,000


## 4. Categorical Attributes

In [6]:
print('=== Seniority Levels ===')
print(df['Seniority'].value_counts())

print(f'\n=== Unique Technologies: {df["Technology"].nunique()} ===')
print(df['Technology'].value_counts().head(15))

print(f'\n=== Unique Countries: {df["Country"].nunique()} ===')
print(df['Country'].value_counts().head(10))

=== Seniority Levels ===
Seniority
Intern       7255
Mid-Level    7253
Trainee      7183
Junior       7100
Architect    7079
Lead         7071
Senior       7059
Name: count, dtype: int64

=== Unique Technologies: 24 ===
Technology
Game Development                     3818
DevOps                               3808
Social Media Community Management    2028
System Administration                2014
Mulesoft                             1973
Development - Backend                1965
Development - FullStack              1961
Adobe Experience Manager             1954
Data Engineer                        1951
Security                             1936
Development - CMS Frontend           1934
Business Intelligence                1934
Database Administration              1933
Client Success                       1927
Design                               1906
Name: count, dtype: int64

=== Unique Countries: 244 ===
Country
Malawi                          242
Spain                           238
Sv

## 5. Numerical Attributes — Scores

In [7]:
score_cols = ['Code Challenge Score', 'Technical Interview Score', 'YOE']
print('=== Descriptive Statistics ===')
print(df[score_cols].describe())

print('\n=== Code Challenge Score Distribution ===')
print(df['Code Challenge Score'].value_counts().sort_index())

print('\n=== Technical Interview Score Distribution ===')
print(df['Technical Interview Score'].value_counts().sort_index())

=== Descriptive Statistics ===
       Code Challenge Score  Technical Interview Score           YOE
count          50000.000000               50000.000000  50000.000000
mean               4.996400                   5.003880     15.286980
std                3.166896                   3.165082      8.830652
min                0.000000                   0.000000      0.000000
25%                2.000000                   2.000000      8.000000
50%                5.000000                   5.000000     15.000000
75%                8.000000                   8.000000     23.000000
max               10.000000                  10.000000     30.000000

=== Code Challenge Score Distribution ===
Code Challenge Score
0     4502
1     4590
2     4579
3     4678
4     4521
5     4479
6     4419
7     4506
8     4619
9     4519
10    4588
Name: count, dtype: int64

=== Technical Interview Score Distribution ===
Technical Interview Score
0     4539
1     4588
2     4500
3     4528
4     4578
5     45

## 6. Application Date Range

In [8]:
df['Application Date'] = pd.to_datetime(df['Application Date'])
print(f'Min date: {df["Application Date"].min()}')
print(f'Max date: {df["Application Date"].max()}')
print(f'\nApplications per year:')
print(df['Application Date'].dt.year.value_counts().sort_index())

Min date: 2018-01-01 00:00:00
Max date: 2022-07-04 00:00:00

Applications per year:
Application Date
2018    11061
2019    11009
2020    11237
2021    11051
2022     5642
Name: count, dtype: int64


## 7. Business Rule — Hiring Outcome

In [9]:
df['is_hired'] = ((df['Code Challenge Score'] >= 7) & (df['Technical Interview Score'] >= 7)).astype(int)
hired = df['is_hired'].sum()
total = len(df)
print(f'HIRED:     {hired:,} ({hired/total*100:.1f}%)')
print(f'NOT HIRED: {total - hired:,} ({(total-hired)/total*100:.1f}%)')
print(f'Overall hiring rate: {hired/total*100:.1f}%')

HIRED:     6,698 (13.4%)
NOT HIRED: 43,302 (86.6%)
Overall hiring rate: 13.4%


## 8. Key Findings Summary

- **Total records:** ~50,000 candidate applications
- **Date range:** 2018–2022
- **Score range:** 0–10 for both assessments
- **Seniority levels:** 7 categories (Trainee to Architect)
- **Technologies:** Multiple domains (Development, QA, Security, DevOps, etc.)
- **Geography:** 100+ countries
- **Hiring rule:** Code Challenge >= 7 AND Technical Interview >= 7